<a href="https://colab.research.google.com/github/JJulianOlivera/educursos/blob/main/2_GradientBoosting_Optimizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install xgboost lightgbm catboost scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. Carga y limpieza
df = pd.read_csv('Churn_Modelling.csv')
df_clean = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
df_encoded = pd.get_dummies(df_clean, columns=['Geography', 'Gender'], drop_first=True)

X = df_encoded.drop(columns=['Exited'])
y = df_encoded['Exited']

# 2. Split Estratificado (Para Churn)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 3. Escalamiento
scaler = StandardScaler()
cols_to_scale = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# 4. Cálculo del desbalance para los modelos
peso_desbalance = (len(y_train) - sum(y_train)) / sum(y_train)

print(f"Datos preparados. Entrenando con {X_train.shape[0]} registros.")

Datos preparados. Entrenando con 8000 registros.


In [10]:
# Función para evaluar y no repetir código
def evaluar_modelo(nombre, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    # PR-AUC es el average_precision_score en scikit-learn
    pr_auc = average_precision_score(y_test, y_proba)

    print(f"\n{nombre}")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}\n")
    return modelo

# 1. Random Forest (Bagging)
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
evaluar_modelo("Random Forest", rf, X_test, y_test)

# 2. XGBoost (Boosting)
xgb = XGBClassifier(scale_pos_weight=peso_desbalance, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
evaluar_modelo("XGBoost Base", xgb, X_test, y_test)

# 3. LightGBM (Optimizado para velocidad)
lgbm = LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
evaluar_modelo("LightGBM", lgbm, X_test, y_test)

# 4. CatBoost (Robusto frente a overfitting)
cb = CatBoostClassifier(auto_class_weights='Balanced', random_state=42, verbose=False)
cb.fit(X_train, y_train)
evaluar_modelo("CatBoost", cb, X_test, y_test)


Random Forest
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.78      0.44      0.56       407

    accuracy                           0.86      2000
   macro avg       0.82      0.70      0.74      2000
weighted avg       0.85      0.86      0.84      2000

ROC-AUC: 0.8508 | PR-AUC: 0.6823


XGBoost Base
              precision    recall  f1-score   support

           0       0.90      0.87      0.89      1593
           1       0.56      0.62      0.59       407

    accuracy                           0.82      2000
   macro avg       0.73      0.75      0.74      2000
weighted avg       0.83      0.82      0.83      2000

ROC-AUC: 0.8358 | PR-AUC: 0.6600


LightGBM
              precision    recall  f1-score   support

           0       0.92      0.84      0.88      1593
           1       0.54      0.73      0.62       407

    accuracy                           0.82      2000
   macro avg       0.

CatBoostClassifier(auto_class_weights='Balanced', random_state=42, verbose=False)

In [12]:
# 1. Define la estrategia de Validación Cruzada
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# 2. Define la grilla de parámetros
param_grid = {
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'n_estimators': [100, 200]
}

# 3. Configura GridSearch priorizando el ROC-AUC
print("Iniciando optimización de XGBoost")
grid_search = GridSearchCV(
    estimator=XGBClassifier(scale_pos_weight=peso_desbalance, random_state=42, eval_metric='logloss'),
    param_grid=param_grid,
    scoring='roc_auc',
    cv=skf,
    verbose=1
)

# Entrena la búsqueda
grid_search.fit(X_train, y_train)

# 4. Extrae el mejor modelo
xgb_optimizado = grid_search.best_estimator_

print(f"Mejores hiperparámetros encontrados")
evaluar_modelo("XGBoost Optimizado", xgb_optimizado, X_test, y_test)

Iniciando optimización de XGBoost
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Mejores hiperparámetros encontrados

XGBoost Optimizado
              precision    recall  f1-score   support

           0       0.93      0.80      0.86      1593
           1       0.50      0.77      0.60       407

    accuracy                           0.79      2000
   macro avg       0.71      0.78      0.73      2000
weighted avg       0.84      0.79      0.81      2000

ROC-AUC: 0.8689 | PR-AUC: 0.7161



XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [13]:
# 1. Define los estimadores base
# Usa los modelos que ya se probaron que funcionan bien
estimadores_base = [
    ('xgb', xgb_optimizado),
    ('lgbm', LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)),
    ('cat', CatBoostClassifier(auto_class_weights='Balanced', random_state=42, verbose=False))
]

# 2. Define el Meta-Learner
meta_learner = LogisticRegression(class_weight='balanced', random_state=42)

# 3. Construye el Stacking Classifier
# cv=5 usa StratifiedKFold para evitar fuga de datos
stacking_model = StackingClassifier(
    estimators=estimadores_base,
    final_estimator=meta_learner,
    cv=5
)

print("Entrenando Ensamble Stacking")
stacking_model.fit(X_train, y_train)

# 4. Evaluación Final
evaluar_modelo("Ensamblado STACKING Final", stacking_model, X_test, y_test)

Entrenando Ensamble Stacking

Ensamblado STACKING Final
              precision    recall  f1-score   support

           0       0.93      0.80      0.86      1593
           1       0.49      0.76      0.60       407

    accuracy                           0.79      2000
   macro avg       0.71      0.78      0.73      2000
weighted avg       0.84      0.79      0.81      2000

ROC-AUC: 0.8689 | PR-AUC: 0.7196



StackingClassifier(cv=5,
                   estimators=[('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='logloss',
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interac...
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=100, n_jobs=None,
                                              num_parallel_tree=None, ...)),
                               ('lgbm',
                                LGBMClassifier(class_weight='balanced',
                                               random_state=42, verbose=-1)),
                               ('cat',
                                CatBoostClassifier(auto_class_weights='Balanced', random_state=42, verbose=False))],
                   final_estimator=LogisticRegression(class_weight='balanced',
                                                      random_state=42))

## Conclusiones del Avance 2: Gradient Boosting y Stacking

### 1. Comparación de Modelos Base y Métricas Especializadas
Al evaluar los algoritmos iniciales, se observa claramente el impacto del desbalanceo de clases:
* **ROC-AUC vs PR-AUC:** Mientras que el ROC-AUC se mantuvo alto en todos los modelos (rondando 0.85 - 0.86), el **PR-AUC** mostró la verdadera dificultad del dataset. LightGBM y CatBoost demostraron un rendimiento superior "fuera de la caja" (PR-AUC de ~0.70) frente al XGBoost base y Random Forest.
* Destaca especialmente el bajo Recall de Random Forest (0.44), demostrando que el Bagging tradicional no fue suficiente para capturar a la clase minoritaria en este contexto.

### 2. Optimización (GridSearchCV en XGBoost)
Se utilizó `StratifiedKFold` para garantizar la integridad de las proporciones de clase, se optimizó los hiperparámetros de XGBoost.
El modelo encontró su punto óptimo con `learning_rate=0.1`, `max_depth=3` y `n_estimators=100`. Esta regularización de la profundidad permitió que el modelo diera un salto de calidad enorme: **el Recall pasó de 0.62 a 0.77**, logrando detectar a la gran mayoría de los clientes en riesgo de fuga.

### 3. El Poder del Ensamble: Stacking
Se implementa una arquitectura de **Stacking** utilizando XGBoost Optimizado, LightGBM y CatBoost, y una Regresión Logística como *Meta-Learner*.
Este ensamble logró la métrica más robusta de todo el experimento: un **PR-AUC de 0.7196** y un **ROC-AUC de 0.8689**. La Regresión Logística aprendió exitosamente en qué modelo base confiar bajo distintas condiciones de datos, entregando una predicción final sumamente estable y superior a cualquier estimador individual.